In [4]:
%pip install pandas matplotlib openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 8.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 8.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 7.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 7.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 7.3 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd

path = "../data/collection/"
files = ["fair_work_rates", "rent", "tax_rates", "lito", "insurance",
         "transport", "living_costs_gov", "food_scenarios",
         "living_costs_other", "source_register"]

def load_table(name):
    """Read a CSV and drop fully empty rows and columns."""
    df = pd.read_csv(path + name + ".csv")
    df = df.dropna(how="all")           # remove empty rows
    df = df.dropna(axis=1, how="all")   # remove empty columns
    return df.reset_index(drop=True)

tables = {name: load_table(name) for name in files}

for name, df in tables.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

fair_work_rates: 22 rows, 5 columns
rent: 15 rows, 6 columns
tax_rates: 8 rows, 6 columns
lito: 4 rows, 6 columns
insurance: 5 rows, 6 columns
transport: 13 rows, 12 columns
living_costs_gov: 5 rows, 11 columns
food_scenarios: 16 rows, 5 columns
living_costs_other: 6 rows, 7 columns
source_register: 14 rows, 10 columns


# Check data types: money columns must be numeric, not text

In [14]:
for name, df in tables.items():
    print("=" * 50)
    print(name)
    print(df.dtypes)

fair_work_rates
award                     str
level                     str
day_type                  str
casual_hourly_rate    float64
effective_from            str
dtype: object
rent
city                str
housing_type        str
weekly_rent     float64
rent_stat           str
source              str
period              str
dtype: object
tax_rates
residency             str
income_from         int64
income_to         float64
base_tax            int64
rate              float64
financial_year        str
dtype: object
lito
income_from              int64
income_to              float64
max_offset               int64
reduction_rate         float64
reduction_starts_at      int64
note                       str
dtype: object
insurance
provider                         str
single_12m_aud               float64
weekly_aud                   float64
source                           str
period                       float64
verified_on_provider_site        str
dtype: object
transport
city            

# Check missing values (per column) and duplicated rows

In [15]:
for name, df in tables.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]
    print(f"{name}: {df.duplicated().sum()} duplicated rows")
    if len(missing):
        print("  missing values:", missing.to_dict())

fair_work_rates: 0 duplicated rows
rent: 0 duplicated rows
tax_rates: 0 duplicated rows
  missing values: {'income_to': 2}
lito: 0 duplicated rows
  missing values: {'income_to': 1}
insurance: 0 duplicated rows
transport: 0 duplicated rows
  missing values: {'concession_eligible': 2, 'valid_from': 12, 'valid_until': 10}
living_costs_gov: 0 duplicated rows
  missing values: {'accommodation': 2, 'transport': 2}
food_scenarios: 0 duplicated rows
living_costs_other: 0 duplicated rows
source_register: 0 duplicated rows


# Check city names are consistent across tables (needed for joins)

In [16]:
expected = {"Sydney", "Melbourne", "Brisbane", "Perth", "Adelaide"}

for name in ["rent", "transport", "living_costs_gov", "food_scenarios"]:
    found = set(tables[name]["city"].str.strip().unique())
    status = "OK" if found == expected else f"MISMATCH: {found ^ expected}"
    print(f"{name}: {status}")

rent: OK
transport: OK
living_costs_gov: OK
food_scenarios: OK


# Sanity checks on values

In [17]:
fw = tables["fair_work_rates"]
rent = tables["rent"]
tr = tables["transport"]

print("Hourly rate range:", fw["casual_hourly_rate"].min(), "-", fw["casual_hourly_rate"].max())
print("Rent range:", rent["weekly_rent"].min(), "-", rent["weekly_rent"].max())
print("Transport base rows per city:")
print(tr[tr["scenario"] == "base"].groupby("city").size())

Hourly rate range: 32.18 - 71.13
Rent range: 331.0 - 759.71
Transport base rows per city:
city
Adelaide     1
Brisbane     1
Melbourne    1
Perth        1
Sydney       1
dtype: int64


# Each city should have exactly one base transport row

In [18]:
assert (tr[tr["scenario"] == "base"].groupby("city").size() == 1).all(), "Transport base rows are not unique"
print("Checks passed")

Checks passed


In [19]:
import os
import pandas as pd

path = "../data/collection/"
clean_path = "../data/clean/"
os.makedirs(clean_path, exist_ok=True)

def load_table(name):
    """Read a CSV; only truly empty cells become missing (so 'N/A' text is kept)."""
    df = pd.read_csv(path + name + ".csv", keep_default_na=False, na_values=[""])
    df = df.dropna(how="all").dropna(axis=1, how="all").reset_index(drop=True)
    # Strip stray whitespace from text columns
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].str.strip()
    return df

tables = {name: load_table(name) for name in files}

# Convert date columns to datetime

In [20]:
date_cols = {
    "fair_work_rates": ["effective_from"],
    "transport": ["valid_from", "valid_until"],
    "living_costs_gov": ["run_date"],
}
for name, cols in date_cols.items():
    for col in cols:
        tables[name][col] = pd.to_datetime(tables[name][col])

# Period in insurance was read as a number; keep it as text

In [21]:
tables["insurance"]["period"] = tables["insurance"]["period"].astype(str)


# Verify the Brisbane 'N/A' issue is fixed

In [30]:
assert tables["transport"]["concession_eligible"].notna().all()
print("Brisbane concession values:",
      tables["transport"].loc[tables["transport"]["city"] == "Brisbane", "concession_eligible"].unique())

Brisbane concession values: <StringArray>
['Not applicable (flat fare for all)']
Length: 1, dtype: str


# Save cleaned tables (the source register stays in data/collection)

In [31]:
for name, df in tables.items():
    if name != "source_register":
        df.to_csv(clean_path + name + ".csv", index=False)

print("Saved:", sorted(os.listdir(clean_path)))

Saved: ['fair_work_rates.csv', 'food_scenarios.csv', 'insurance.csv', 'lito.csv', 'living_costs_gov.csv', 'living_costs_other.csv', 'rent.csv', 'tax_rates.csv', 'transport.csv']
